# Lab 08 — Distributed Training and Sharding Simulation
**Goal:** Simulate data parallelism and parameter sharding to understand memory and communication without requiring multiple GPUs.

Use the loop: predict → run → inspect → break → decide.


In [ ]:
import numpy as np
n_params=1_000_000_000
ranks=8
bytes_per_param=2
print("replicated weights GiB:", n_params*bytes_per_param/1024**3)


## Prediction
Predict how DDP and sharding differ in GPU memory per rank for a 1B-parameter model.


In [ ]:
rep=n_params*bytes_per_param
sharded=rep/ranks
print("8-way shard GiB:",sharded/1024**3)
print("replicated -> sharded reduction:",rep/sharded)


## Interpretation
DDP replicates model state on each rank; sharding reduces per-rank residency but increases communication/coordination complexity.


In [ ]:
# Communication thought experiment: every rank needs gradients synchronized in classic DDP.
# FSDP-style sharding changes what is resident on each rank and introduces all-gather/reduce-scatter patterns.
tensor_bytes=512*1024**2
print("example tensor MiB:",tensor_bytes/1024**2)
print("8 ranks, ideal equal shard MiB:",tensor_bytes/8/1024**2)


## Answers
FSDP2 shards parameters, gradients and optimizer state so models that do not fit on a single accelerator can become trainable. The tradeoff is communication and implementation complexity, so benchmark the complete training step.

### Primary source
https://docs.pytorch.org/tutorials/intermediate/FSDP_tutorial.html


## Deepening: replication versus sharding

This section upgrades the notebook from a runnable demo to a measurable experiment. Predict before executing.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
N = 7_000_000_000
weight_bytes_bf16 = N * 2

for k in [1, 2, 4, 8]:
    replicated = weight_bytes_bf16
    sharded = weight_bytes_bf16 / k
    print(f"{k} GPU(s): replicated={replicated/1e9:.1f} GB, sharded={sharded/1e9:.1f} GB")

print("Sharding parameters does not eliminate optimizer, activation, communication, or runtime costs.")


## Break experiment

Assume communication consumes an increasing fraction of step time as GPU count rises.

Explain why ideal linear scaling is only an upper bound.

### Required table

GPUs → throughput → scaling efficiency → communication fraction → memory/GPU

Use templates/EXPERIMENT_CARD.md to record the run.


## 7. Research exercise — derive a distributed workload from real data


In [ ]:
from datasets import load_dataset
import numpy as np

ds=load_dataset("HuggingFaceFW/fineweb_100BT",split="train",streaming=True)
tokens=[]
for row in ds:
    if row.get("token_count"): tokens.append(int(row["token_count"]))
    if len(tokens)>=1000: break

tokens=np.array(tokens)
target_tokens=8_000_000
for world in [1,2,4,8]:
    global_batch=target_tokens
    per_rank=global_batch/world
    print(f"world_size={world} tokens/rank/step={per_rank:,.0f}")
print("document token p50/p95:",np.percentile(tokens,[50,95]).astype(int))


<details><summary><strong>Expected findings / answer</strong></summary>

If the global token budget per step is held constant, increasing world size reduces the token work assigned to each rank. That can improve throughput only if communication and synchronization do not dominate. Real document lengths also make packing important: uneven batches can create stragglers. The correct scaling experiment therefore measures **tokens/sec, synchronization/communication time, and load balance**, not GPU count alone.
</details>
